# LUT Dependence Visualization

Explore which FPGA-build configuration parameters drive `real_LUT` (real, post-synthesis LUT usage), as a first step toward picking a regression model for `hardware/mvau_lut_calibration_dataset.csv`.

For every plot below, **y = `real_LUT`**. The x-axis panel shows each configuration parameter **independently** (one miniature scatter plot per parameter), not yet combined into a joint model.

The loading/plotting code is written generically (`DATASET_PATH`, `plot_param_panel`) so the same notebook can later be pointed at other calibration CSVs with the same schema (e.g. an alpha025-specific dataset) just by changing `DATASET_PATH` and re-running.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def _find_repo_root(start: Path) -> Path:
    """Walk up from cwd looking for the repo root, so this notebook works the
    same whether Jupyter's cwd is the repo root or analysis/hardware_calibration/ itself."""
    for candidate in [start, *start.parents]:
        if (candidate / "hardware").exists() and (candidate / "analysis").exists():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())

# Point this at a different calibration CSV (same schema) to reuse the whole notebook.
DATASET_PATH = REPO_ROOT / "hardware" / "mvau_lut_calibration_dataset.csv"

sns.set_theme(style="whitegrid")

## Load Dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f"loaded {DATASET_PATH.relative_to(REPO_ROOT)}: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Inspect Dataset Structure

Column dtypes and a check for missing values, before deciding which columns are usable numeric features vs. categorical ones.

In [ ]:
print(df.dtypes)

missing = df.isna().sum()
missing = missing[missing > 0]
print("\nmissing values per column:" if len(missing) else "\nno missing values")
missing

## Identify Feature and Target Columns

Target is `real_LUT`. Features are split into the numeric build-config parameters and the categorical/boolean ones. Note the other `real_*` columns (`real_LUTRAM`, `real_FF`, `real_DSP`, ...) are excluded from the feature list -- they're synthesis *outputs*, like `real_LUT` itself, not configuration inputs.

In [ ]:
Y_COL = "real_LUT"

NUMERIC_PARAMS = [
    "partition", "MH", "MW", "PE", "SIMD",
    "PE_times_SIMD", "log2_MW", "weight_bits", "act_bits",
]
CATEGORICAL_PARAMS = [
    "op_type", "weightDataType", "inputDataType", "outputDataType",
    "resType", "force_dsp", "ram_style", "mem_mode", "pe_simd_landed_matches_json",
]
ALL_PARAMS = NUMERIC_PARAMS + CATEGORICAL_PARAMS

print(f"target: {Y_COL}")
print(f"{len(NUMERIC_PARAMS)} numeric params, {len(CATEGORICAL_PARAMS)} categorical params")

## Compute Summary Statistics

In [ ]:
df[NUMERIC_PARAMS + [Y_COL]].describe().T

## Create Scatter Plot Panel (Feature vs LUT)

`plot_param_panel` builds one miniature scatter plot per parameter, always with `real_LUT` on the y-axis (log-scaled by default, since it spans multiple orders of magnitude):
- **Numeric** params (`MH`, `PE`, `weight_bits`, ...) get a plain scatter plus a linear trend line and a Pearson `r` annotation (see next two sections).
- **Categorical/boolean** params (`ram_style`, `op_type`, ...) get their category codes jittered slightly on the x-axis, with the real category names as tick labels.

Every point is colored by `op_type` (e.g. `MVAU_hls` vs `VVAU_hls`), with a single shared legend for the whole figure.

In [ ]:
def plot_param_panel(
    df: pd.DataFrame,
    params: list[str],
    y_col: str = Y_COL,
    y_log: bool = True,
    ncols: int = 4,
    add_trend: bool = True,
    title: str | None = None,
) -> plt.Figure:
    """Grid of miniature scatter plots, one per parameter in `params`, always
    plotting df[y_col] on the y-axis. Numeric params (per NUMERIC_PARAMS) get
    a linear trend line + Pearson r in the subplot title; everything else is
    treated as categorical: jittered integer category codes on the x-axis,
    with the real category names as tick labels. Every point is colored by
    `op_type`, with one shared legend for the whole figure."""
    op_types = sorted(df["op_type"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    color_map = {op: cmap(i) for i, op in enumerate(op_types)}
    colors = df["op_type"].astype(str).map(color_map)

    nrows = int(np.ceil(len(params) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), squeeze=False)
    axes = axes.ravel()
    rng = np.random.default_rng(0)
    y = df[y_col].astype(float)

    for ax, param in zip(axes, params):
        if param in NUMERIC_PARAMS:
            x = df[param].astype(float)
            ax.scatter(x, y, c=colors, s=14, alpha=0.75, edgecolors="none")
            if add_trend and x.nunique() > 1:
                coeffs = np.polyfit(x, y, deg=1)
                x_line = np.linspace(x.min(), x.max(), 50)
                ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
            r = np.corrcoef(x, y)[0, 1] if x.nunique() > 1 else float("nan")
            ax.set_title(f"{param}  (r={r:.2f})", fontsize=9)
        else:
            cats = sorted(df[param].astype(str).unique())
            code_of = {c: i for i, c in enumerate(cats)}
            codes = df[param].astype(str).map(code_of).astype(float)
            jitter = rng.uniform(-0.15, 0.15, size=len(codes))
            ax.scatter(codes + jitter, y, c=colors, s=14, alpha=0.75, edgecolors="none")
            ax.set_xticks(range(len(cats)))
            ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=7)
            ax.set_title(param, fontsize=9)

        ax.set_xlabel(param, fontsize=8)
        ax.tick_params(labelsize=7)
        if y_log:
            ax.set_yscale("log")

    for ax in axes[len(params):]:
        ax.axis("off")
    for row in range(nrows):
        axes[row * ncols].set_ylabel(y_col, fontsize=8)

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op)
        for op in op_types
    ]
    fig.legend(handles=handles, loc="upper right", fontsize=9, title="op_type")

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_param_panel(
    df, ALL_PARAMS,
    title=f"{DATASET_PATH.name}: real_LUT vs each parameter",
)
plt.show()

## Highlight Correlation Strength per Feature

The trend lines/`r` values above are read off each subplot title; here they're collected into one sorted table (by `|r|`, strongest first) for easier comparison across all numeric parameters at once.

In [ ]:
y = df[Y_COL].astype(float)
corr_rows = []
for param in NUMERIC_PARAMS:
    x = df[param].astype(float)
    r = np.corrcoef(x, y)[0, 1] if x.nunique() > 1 else float("nan")
    corr_rows.append({"parameter": param, "pearson_r": r, "abs_r": abs(r)})

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r")
    .reset_index(drop=True)
)
corr_df

## Save Visualizations

In [ ]:
RESULTS_DIR = REPO_ROOT / "analysis" / "hardware_calibration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

out_path = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_panel.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"saved panel to {out_path.relative_to(REPO_ROOT)}")

## Reusing this notebook for other calibration CSVs

`plot_param_panel`, `NUMERIC_PARAMS`/`CATEGORICAL_PARAMS`, and the correlation-table cell all work off `df` alone -- to analyze another calibration dataset with the same schema (e.g. `hardware/mvau_lut_calibration_dataset_12_separable_dense_relu_alpha025.csv`), just change `DATASET_PATH` in the Load Dataset cell above, re-run from there, and everything else (structure inspection, panel plot, correlation table, saved PNG) regenerates for the new dataset.